EDA

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

In [ ]:
dir = Path('Data') 
 
files = [
    'customers', 'geography', 'inventory', 'order_items', 'orders',
    'payments', 'products', 'promotions', 'returns', 'reviews',
    'sales', 'shipments', 'web_traffic', 'sample_submission'
]

dfs = {}

for file in files:
    dfs[file] = pd.read_csv(dir / f'{file}.csv')



df_sales = dfs['sales']
df_promos = dfs['promotions']
df_traffic = dfs['web_traffic'].rename(columns={'date': 'Date'})
submission = dfs['sample_submission']

FileNotFoundError: [Errno 2] No such file or directory: 'datathon-2026-round-1\\customers.csv'

Features extraction

In [ ]:
%pip install holidays -q

import holidays

Note: you may need to restart the kernel to use updated packages.


In [ ]:
def parse_date_col(df, col):
    df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed')
    if df[col].isna().any():
        bad_rows = df.loc[df[col].isna()].head()
        raise ValueError(f"Could not parse {col} as dates. Examples:\n{bad_rows}")
    return df

df_sales = parse_date_col(df_sales, 'Date')
submission = parse_date_col(submission, 'Date')
df_traffic = parse_date_col(df_traffic, 'Date')

df_test = submission[['Date']].copy()
df_test['Revenue'] = np.nan
df_test['COGS'] = np.nan

df_master = pd.concat([df_sales[['Date', 'Revenue', 'COGS']], df_test], ignore_index=True)
df_master = parse_date_col(df_master, 'Date')

if df_master['Date'].isna().any():
    bad_rows = df_master.loc[df_master['Date'].isna()].head()
    raise ValueError(f"Date conversion failed for {df_master['Date'].isna().sum()} rows. Examples:\n{bad_rows}")

df_master = df_master.sort_values('Date').reset_index(drop=True)

df_master['year'] = df_master['Date'].dt.year
df_master['month'] = df_master['Date'].dt.month
df_master['day'] = df_master['Date'].dt.day
df_master['dayofweek'] = df_master['Date'].dt.dayofweek
df_master['dayofyear'] = df_master['Date'].dt.dayofyear

# PROMOTIONS & FOMO
df_promos = parse_date_col(df_promos, 'start_date')
df_promos = parse_date_col(df_promos, 'end_date')

if df_promos[['start_date', 'end_date']].isna().any().any():
    bad_promos = df_promos[df_promos[['start_date', 'end_date']].isna().any(axis=1)].head()
    raise ValueError(f"Promotion date conversion failed. Examples:\n{bad_promos}")

df_promos['promo_duration'] = (df_promos['end_date'] - df_promos['start_date']).dt.days + 1
df_promos['active_dates'] = df_promos.apply(lambda row: pd.date_range(row['start_date'], row['end_date']).tolist(), axis=1)
promo_daily = df_promos.explode('active_dates').rename(columns={'active_dates': 'Date'})

promo_features = promo_daily.groupby('Date').agg(
    Promo_Count=('promo_id', 'nunique'),
    Max_Discount=('discount_value', 'max'),
    Min_Duration=('promo_duration', 'min')
).reset_index()

promo_starts = df_promos[['start_date']].drop_duplicates().rename(columns={'start_date': 'Date'})
promo_starts['is_promo_start'] = 1
promo_ends = df_promos[['end_date']].drop_duplicates().rename(columns={'end_date': 'Date'})
promo_ends['is_promo_end'] = 1

df_master = df_master.merge(promo_features, on='Date', how='left').fillna({'Promo_Count': 0, 'Max_Discount': 0, 'Min_Duration': 30})
df_master = df_master.merge(promo_starts, on='Date', how='left').fillna({'is_promo_start': 0})
df_master = df_master.merge(promo_ends, on='Date', how='left').fillna({'is_promo_end': 0})

df_master['Is_Promo'] = (df_master['Promo_Count'] > 0).astype(int)
df_master['is_promo_stacked'] = (df_master['Promo_Count'] >= 2).astype(int)
df_master['is_golden_promo'] = (df_master['Promo_Count'] == 2).astype(int)

# WEB TRAFFIC
traffic_daily = df_traffic.groupby('Date').agg(
    Total_Sessions=('sessions', 'sum'),
    Avg_Duration=('avg_session_duration_sec', 'mean'),
    Avg_Bounce=('bounce_rate', 'mean')
).reset_index()

traffic_daily['Intent_Index'] = traffic_daily['Avg_Duration'] * (1 - traffic_daily['Avg_Bounce'])
df_master = df_master.merge(traffic_daily[['Date', 'Total_Sessions', 'Intent_Index']], on='Date', how='left')

df_master['Traffic_Trend'] = df_master['Total_Sessions'].rolling(window=14, min_periods=1).mean().bfill().ffill()
df_master['Intent_Trend'] = df_master['Intent_Index'].rolling(window=14, min_periods=1).mean().bfill().ffill()

# CUSTOMERS
try:
    df_cust = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['signup_date'])
    daily_signups = df_cust.groupby('signup_date').size().reset_index(name='New_Users').rename(columns={'signup_date': 'Date'})
    df_master = df_master.merge(daily_signups, on='Date', how='left').fillna({'New_Users': 0})
    avg_new_users = df_master[df_master['Date'].dt.year == 2022]['New_Users'].mean()
    df_master.loc[df_master['Date'].dt.year >= 2023, 'New_Users'] = avg_new_users
    df_master['Total_Users'] = df_master['New_Users'].cumsum()
except:
    df_master['Total_Users'] = 0

# REVIEWS & RETURNS
try:
    df_reviews = pd.read_csv(DATA_DIR + 'reviews.csv', parse_dates=['review_date'])
    daily_reviews = df_reviews.groupby('review_date').agg(Avg_Rating=('rating', 'mean')).reset_index().rename(columns={'review_date': 'Date'})
    df_master = df_master.merge(daily_reviews, on='Date', how='left')
    df_master['Avg_Rating'] = df_master['Avg_Rating'].rolling(window=14, min_periods=1).mean().bfill().ffill()
except:
    df_master['Avg_Rating'] = 4.0

try:
    df_returns = pd.read_csv(DATA_DIR + 'returns.csv', parse_dates=['return_date'])
    daily_returns = df_returns.groupby('return_date').agg(Refund_Amount=('refund_amount', 'sum')).reset_index().rename(columns={'return_date': 'Date'})
    df_master = df_master.merge(daily_returns, on='Date', how='left').fillna({'Refund_Amount': 0})
except:
    df_master['Refund_Amount'] = 0


df_master['Is_Weekend'] = df_master['dayofweek'].isin([5, 6]).astype(int)
df_master['is_peak_weekday'] = df_master['dayofweek'].isin([0, 1]).astype(int)
df_master['is_month_end'] = df_master['Date'].dt.is_month_end.astype(int)
df_master['is_payday'] = df_master['day'].isin([28, 29, 30, 31, 1, 2, 3, 4, 5]).astype(int)
df_master['is_high_season'] = df_master['month'].isin([3, 4, 5]).astype(int)
df_master['is_double_day'] = (df_master['month'] == df_master['day']).astype(int)

df_master['is_pre_double_day'] = df_master['is_double_day'].shift(-1).fillna(0).astype(int)
df_master['is_post_double_day'] = df_master['is_double_day'].shift(1).fillna(0).astype(int)
df_master['is_pre_promo'] = df_master['is_promo_start'].shift(-1).fillna(0).astype(int)
df_master['is_post_promo'] = df_master['is_promo_end'].shift(1).fillna(0).astype(int)
df_master['payday_promo_start'] = df_master['is_payday'] * df_master['is_promo_start']

df_master['revenue_lag_364'] = df_master['Revenue'].shift(364)
df_master['revenue_lag_364'] = df_master['revenue_lag_364'].ffill().fillna(df_master['Revenue'].mean())
df_master['revenue_lag_364_mean_7'] = df_master['revenue_lag_364'].rolling(window=7, center=True, min_periods=1).mean()

vn_holidays = holidays.VN(years=range(2012, 2025))
df_master['Is_Holiday'] = df_master['Date'].dt.date.isin(vn_holidays.keys()).astype(int)

# BASELINE
train_only = df_master[df_master['year'] < 2023].copy()
annual_means = train_only.groupby('year')['Revenue'].transform('mean')
train_only['rev_norm'] = train_only['Revenue'] / annual_means
seasonal_profile = train_only.groupby(['month', 'day'])['rev_norm'].mean().reset_index()

df_master = df_master.merge(seasonal_profile, on=['month', 'day'], how='left').fillna({'rev_norm': 1.0})
base_rev_2022 = train_only[train_only['year'] == 2022]['Revenue'].mean()
df_master['years_ahead'] = np.maximum(0, df_master['year'] - 2022)
df_master['Baseline_Revenue'] = base_rev_2022 * (1.17 ** df_master['years_ahead']) * df_master['rev_norm'] ## predefine 17% growth (can adjust or think of a better way to predict yearly growth)

def encode_cyclical(df, col, max_val):
    df[col + '_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[col + '_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df
df_master = encode_cyclical(df_master, 'month', 12.0)
df_master = encode_cyclical(df_master, 'dayofweek', 7.0)
df_master = encode_cyclical(df_master, 'dayofyear', 366.0)

# LOG TRANSFORMS
df_master['log_Baseline'] = np.log1p(df_master['Baseline_Revenue'])
df_master['log_Lag_364'] = np.log1p(df_master['revenue_lag_364'])
df_master['log_Lag_7'] = np.log1p(df_master['revenue_lag_364_mean_7'])

In [ ]:
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4381 entries, 0 to 4380
Data columns (total 49 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    4381 non-null   datetime64[ns]
 1   Revenue                 3833 non-null   float64       
 2   COGS                    3833 non-null   float64       
 3   year                    4381 non-null   int32         
 4   month                   4381 non-null   int32         
 5   day                     4381 non-null   int32         
 6   dayofweek               4381 non-null   int32         
 7   dayofyear               4381 non-null   int32         
 8   Promo_Count             4381 non-null   float64       
 9   Max_Discount            4381 non-null   float64       
 10  Min_Duration            4381 non-null   float64       
 11  is_promo_start          4381 non-null   float64       
 12  is_promo_end            4381 non-null   float64 

Prediction model

LightGBM

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import numpy as np

features = [
    'year', 'Is_Weekend', 'Is_Holiday', 'is_month_end', 'is_payday', 'is_high_season',
    'is_double_day', 'is_peak_weekday',
    'is_pre_double_day', 'is_post_double_day', 'is_pre_promo', 'is_post_promo', 'payday_promo_start',
    'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos',
    'Promo_Count', 'Max_Discount', 'Min_Duration', 'Is_Promo', 'is_promo_stacked', 'is_golden_promo',
    'is_promo_start', 'is_promo_end',
    'Traffic_Trend', 'Intent_Trend',
    'Total_Users', 'Avg_Rating', 'Refund_Amount',
    'log_Baseline', 'log_Lag_364', 'log_Lag_7'
]

train_mask = df_master['Revenue'].notna()
test_mask = df_master['Revenue'].isna()

X_train = df_master.loc[train_mask, features].copy()
X_test = df_master.loc[test_mask, features].copy()
y_train_log = np.log1p(df_master.loc[train_mask, 'Revenue'])

tscv = TimeSeriesSplit(n_splits=3)
base_model = LGBMRegressor(
    objective='rmse',
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

param_grid = {
    'n_estimators': [800, 1500],
    'learning_rate': [0.01, 0.03],
    'max_depth': [5, 7],
    'num_leaves': [31, 63],
    'min_child_samples': [20, 40],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train_log)

best_params = grid_search.best_params_
print('Best params:', best_params)
print('Best CV RMSE:', -grid_search.best_score_)

seeds = [42, 123, 999, 2024, 8888]
models = []

for seed in seeds:
    model = LGBMRegressor(
        **best_params,
        objective='rmse',
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )
    model.fit(X_train, y_train_log)
    models.append(model)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 5, 'min_child_samples': 40, 'n_estimators': 800, 'num_leaves': 31, 'subsample': 0.8}
Best CV RMSE: 0.3371704979706303


Evaluation

In [ ]:


preds_log_list = [model.predict(X_test) for model in models]
test_preds_log_avg = np.mean(preds_log_list, axis=0)

test_preds_real = np.expm1(test_preds_log_avg)
submission["Revenue"] = np.maximum(0, test_preds_real)

avg_cogs_ratio = (df_sales["COGS"] / df_sales["Revenue"]).mean()
submission["COGS"] = submission["Revenue"] * avg_cogs_ratio



output_path = "submission_v3.csv"
submission.to_csv(output_path, index=False)
print("Saved:", output_path)

Saved: submission_lightgbm_gridsearch_ensemble.csv
